In [0]:
def readBronzeOnEv():
    from pyspark.sql import functions as F
    print("Beginning to read from Bronze")
    print(".")
    print(".")
    df_on_silver= (spark.readStream
                    .table("ev_spark.bronze.ontario_ev")
                    .withColumnRenamed("FSA", "fsa")
                    .withColumn("ev_count", F.col("Total_EV").cast("int"))
                    .withColumn("province", F.lit("Ontario"))
                    .select("fsa", "province", "ev_count")
                    )
    print("Read Successful")
    print("**************************")
    return df_on_silver

In [0]:
def writeToSilver(df):
    print("Writing to silver")
    print(".")
    print(".")
    (df.writeStream
                    .format("delta")
                    .option("checkpointLocation", "/Volumes/ev_spark/myvol/checkpoint/chkpt/ontario_ev_silver/")
                    .outputMode("append")
                    .trigger(availableNow=True)
                    .table("ev_spark.silver.ontario_ev")
    )
    print("Write successful")
    print("****************************")

In [0]:
readOnEv_df = readBronzeOnEv()
writeToSilver(readOnEv_df)